# 03 · Дополнительные поля 2ГИС для кофеен

Поиск 2ГИС (основной сбор и догрузка) сохраняет о заведении только ID, название, координаты, рейтинг и число отзывов. Здесь мы догружаем по ID поля, которые важны для выбора места кофейни. Это нужно для заведений, найденных текстовым поиском. Признаки кандидатов по всем кофейням города считает `03_2gis_all_coffee.ipynb`: там эти поля приходят сразу в поисковых запросах.

Догружаемые поля:
- **рубрики** — настоящая кофейня или кондитерская или ресторан: поиск «кофейня» приносит и их;
- **сеть или одиночное заведение** — число филиалов организации и бренда. Сети сами выбирают места по исследованиям, так что их точки подтверждают спрос;
- **часы работы** — открыта ли к 8:00 (утренний поток на работу), круглосуточно, по выходным;
- **временно закрытые филиалы** — их не считаем конкурентами;
- **атрибуты** — средний чек (ценовой сегмент), число мест, «кофе с собой», Wi-Fi, «можно с ноутбуком»;
- **даты карточки** и **здание**, если 2ГИС его указывает.

Запрос `/3.0/items/byid` принимает до 100 ID, поэтому на ~1 300 заведений уходит около 13–15 запросов. Ответы 2ГИС хранятся в SQLite целиком, так что разбор и признаки можно пересчитывать без новых запросов.

**Входы**
- Файлы прогресса поиска: основной сбор `data/processed/2gis_almaty_progress.sqlite3` (из `03_2gis_data_safe.ipynb`, он у Давида) и догрузка `data/processed/2gis_gap/gap_progress.sqlite3` (из `03_2gis_gap_fill.ipynb`). Отсутствующий файл пропускается.
- Ключ `DGIS_API_KEY` в окружении или в `.env`. Он нужен только для запросов.

**Выходы**
- `data/processed/2gis_enriched/enrich_progress.sqlite3` — ответы byid и число HTTP-попыток.
- `data/processed/2gis_enriched/organizations_enriched.csv` — по строке на заведение; `id` хранится текстом.

С настройками по умолчанию ноутбук не обращается к 2ГИС.

In [1]:
from contextlib import closing
import json
from pathlib import Path
import sqlite3
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = (Path("..") if Path.cwd().name == "notebooks" else Path(".")).resolve()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from dgis_api import (  # общий клиент 2ГИС: notebooks/dgis_api.py
    BYID_MAX_IDS, BYID_URL, ENRICH_FIELDS, CollectionStopped, attribute_number, classify, enrich, load_enriched,
    parse_item, parse_schedule, read_api_key,
)

SEARCH_SOURCES = {
    "main": PROJECT_ROOT / "data" / "processed" / "2gis_almaty_progress.sqlite3",       # основной сбор (Давид)
    "gap": PROJECT_ROOT / "data" / "processed" / "2gis_gap" / "gap_progress.sqlite3",   # догрузка 18 точек
}
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "2gis_enriched"
ENRICH_CHECKPOINT = OUT_DIR / "enrich_progress.sqlite3"
ENRICHED_CSV = OUT_DIR / "organizations_enriched.csv"

COFFEE_RUBRICS = {"Кофейни"}        # какие рубрики 2ГИС считаем кофейней; проверьте по таблице рубрик после пробного пакета
CHAIN_MIN_BRANCHES = 3              # сеть — от 3 филиалов у организации или бренда

## Что уже найдено поиском

Из каждого файла прогресса читаем найденные заведения и центры поиска с их статусами. Одно заведение могло найтись в обоих сборах, поэтому объединяем по `id`, а в столбце `source` указываем, где оно нашлось.

In [2]:
def read_search_source(name, path):
    with closing(sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)) as db:
        orgs = pd.read_sql_query("SELECT id, name, lon, lat, general_rating, general_review_count FROM organizations", db)
        centers = pd.read_sql_query("SELECT point, status FROM centers", db)
        plan = json.loads(db.execute("SELECT plan FROM run").fetchone()[0])
    orgs["source"] = name
    centers["source"] = name
    centers["radius_m"] = plan["params"]["radius"]
    centers["query"] = plan["params"]["q"]
    return orgs, centers


found, search_centers = None, None
parts = [read_search_source(name, path) for name, path in SEARCH_SOURCES.items() if path.is_file()]
for name, path in SEARCH_SOURCES.items():
    print(f"{name}: {path.relative_to(PROJECT_ROOT)} — {'есть' if path.is_file() else 'нет, пропускаем'}")
if parts:
    found = pd.concat([o for o, _ in parts], ignore_index=True).sort_values(["id", "source"])
    found = found.groupby("id", as_index=False).agg(
        name=("name", "first"), lon=("lon", "first"), lat=("lat", "first"),
        source=("source", lambda s: ";".join(sorted(set(s)))),
    )
    search_centers = pd.concat([c for _, c in parts], ignore_index=True)
    print(f"Уникальных заведений: {len(found)}; по источникам: {found['source'].value_counts().to_dict()}")
    print("Статусы центров поиска:", search_centers.groupby("source")["status"].value_counts().to_dict())
    capped = (search_centers["status"] == "capped").sum()
    if capped:
        print(f"Внимание: {capped} центров упёрлись в предел страниц (capped) — рядом с ними собраны не все заведения.")
else:
    print("Файлов поиска нет. Попросите у Давида data/processed/2gis_almaty_progress.sqlite3 "
          "и/или запустите 03_2gis_gap_fill.ipynb.")

main: data\processed\2gis_almaty_progress.sqlite3 — нет, пропускаем
gap: data\processed\2gis_gap\gap_progress.sqlite3 — есть
Уникальных заведений: 7; по источникам: {'gap': 7}
Статусы центров поиска: {('gap', 'complete'): 14}


## Офлайн-тесты

Ответы 2ГИС подменяются, сеть и квота не используются. Тесты проверяют:
- пакеты по ID и ненайденные ID;
- что при добавлении новых ID уже загруженные не запрашиваются повторно;
- деление пакета, на который 2ГИС ответил «не найдено»;
- что ключ не попадает в сообщения об ошибках;
- бюджет попыток и продолжение после остановки;
- разбор ответа по образцу из документации 2ГИС;

In [3]:
import tempfile
import unittest
from unittest.mock import Mock


def byid_response(items=(), *, code=200, error=None, status=200):
    data = {"meta": {"code": code}, "result": {"items": list(items), "total": len(items)}}
    if error:
        data["meta"]["error"] = error
    return Mock(status_code=status, headers={"Retry-After": None}, **{"json.return_value": data})


SAMPLE_ITEM = {  # поля и вложенность — по образцам ответа в документации 2ГИС
    "id": "70000001029535674_7a1b", "name": "Coffee Point, кофейня",
    "point": {"lon": 76.945, "lat": 43.238},
    "rubrics": [{"id": "164", "name": "Кондитерские", "kind": "additional"},
                {"id": "162", "name": "Кофейни", "kind": "primary"}],
    "org": {"branch_count": 7, "id": "70000001087490185", "name": "Coffee Point"},
    "brand": {"branch_count": 14, "id": "70000001103777439", "name": "Coffee Point"},
    "reviews": {"general_rating": 4.7, "general_review_count": 32},
    "schedule": {"Mon": {"working_hours": [{"from": "07:30", "to": "22:00"}]},
                 "Tue": {"working_hours": [{"from": "09:00", "to": "22:00"}]},
                 "Sat": {"working_hours": [{"from": "10:00", "to": "03:30"}]}},
    "flags": {"temporary_closed": "string", "photos": True},
    "address": {"building_name": "Нурлы Тау, бизнес-центр", "postcode": "050000"},
    "attribute_groups": [  # теги и названия — как в реальных ответах 2ГИС по Алматы
        {"name": "Ресторан / Кафе", "attributes": [{"id": "1", "name": "Чек 4500\xa0₸", "tag": "food_service_avg_price"},
                                                   {"id": "2", "name": "До 20\xa0мест", "tag": "food_service_capacity"},
                                                   {"id": "3", "name": "Можно с ноутбуком", "tag": "food_service_cafe_working"}]},
        {"name": "Напитки", "attributes": [{"id": "4", "name": "Кофе с собой", "tag": "cocktail_bars_types_coffee_take_away"}]},
    ],
    "dates": {"created_at": "2012-04-18T12:08:03.273144+07:00", "updated_at": "2012-06-18T11:43:03.273144+07:00"},
}


class EnrichTests(unittest.TestCase):
    def setUp(self):
        temp = tempfile.TemporaryDirectory()
        self.addCleanup(temp.cleanup)
        self.checkpoint = Path(temp.name) / "enrich.sqlite3"
        self.clock = 1_700_000_000.0

    def sleep(self, seconds):
        self.clock += seconds

    def run_enrich(self, ids, responses, *, limit=20, batch_size=2, fields=ENRICH_FIELDS, batch_limit=None, key="mock-credential"):
        self.http = Mock(side_effect=responses)
        result = enrich(ids, self.checkpoint, key=key, max_requests_total=limit, get=self.http, fields=fields,
                        batch_size=batch_size, batch_limit=batch_limit, now=lambda: self.clock, sleep=self.sleep)
        self.assertTrue(all(call.args == (BYID_URL,) for call in self.http.call_args_list))
        return result

    def requested(self):
        return [call.kwargs["params"]["id"] for call in self.http.call_args_list]

    def test_batches_missing_ids_and_params(self):
        result = self.run_enrich(["105", "101", "103", "102", "104"], [
            byid_response([{"id": "101"}, {"id": "102"}]),
            byid_response([{"id": "103"}]),
            byid_response([{"id": "105_hash"}]),
        ])
        self.assertEqual(self.requested(), ["101,102", "103,104", "105"])
        self.assertEqual(self.http.call_args.kwargs["params"]["fields"], ",".join(ENRICH_FIELDS))
        self.assertEqual((result["items"], result["missing"], result["attempts"]), (4, 1, 3))
        self.assertEqual(result["batches"]["done"], 3)
        self.assertIsNone(result["stop_reason"])
        _, missing, _ = load_enriched(self.checkpoint)
        self.assertEqual(missing, {"104"})

    def test_new_ids_are_appended_without_refetch(self):
        self.run_enrich(["1", "2"], [byid_response([{"id": "1"}, {"id": "2"}])])
        result = self.run_enrich(["1", "2", "3"], [byid_response([{"id": "3"}])])
        self.assertEqual(self.requested(), ["3"])
        self.assertEqual((result["items"], result["attempts"]), (3, 2))

    def test_changed_fields_are_rejected(self):
        self.run_enrich(["1"], [byid_response([{"id": "1"}])])
        with self.assertRaisesRegex(CollectionStopped, "Список полей изменился"):
            self.run_enrich(["1"], [], fields=("items.point",))

    def test_empty_batch_is_split(self):
        result = self.run_enrich(["1", "2", "3", "4"], [
            byid_response(code=404, error={"type": "itemNotFound"}),
            byid_response([{"id": "1"}, {"id": "2"}]),
            byid_response([{"id": "3"}]),
        ], batch_size=4)
        self.assertEqual(self.requested(), ["1,2,3,4", "1,2", "3,4"])
        self.assertEqual((result["items"], result["missing"]), (3, 1))
        self.assertEqual(result["batches"], {"pending": 0, "done": 2, "split": 1})

    def test_bad_ids_are_rejected(self):
        with self.assertRaises(ValueError):
            self.run_enrich(["12a"], [])

    def test_server_error_message_hides_key(self):
        result = self.run_enrich(["1"], [byid_response(code=403, error={"message": "Field not allowed for key mock-credential"})])
        self.assertIn("meta.code 403", result["stop_reason"])
        self.assertIn("Field not allowed", result["stop_reason"])
        self.assertNotIn("mock-credential", result["stop_reason"])
        self.assertEqual(result["batches"]["pending"], 1)

    def test_budget_stop_and_resume(self):
        result = self.run_enrich(["1", "2", "3"], [byid_response([{"id": "1"}, {"id": "2"}])], limit=1)
        self.assertIn("лимит", result["stop_reason"])
        self.assertEqual(result["batches"]["pending"], 1)
        result = self.run_enrich(["1", "2", "3"], [byid_response([{"id": "3"}])], limit=2)
        self.assertEqual(self.requested(), ["3"])
        self.assertEqual((result["items"], result["attempts"], result["batches"]["pending"]), (3, 2, 0))

    def test_batch_limit(self):
        result = self.run_enrich(["1", "2", "3"], [byid_response([{"id": "1"}, {"id": "2"}])], batch_limit=1)
        self.assertEqual(result["batches"]["pending"], 1)

    def test_parse_sample_item(self):
        row = parse_item(SAMPLE_ITEM)
        self.assertEqual(row["id"], "70000001029535674")
        self.assertEqual((row["rubric_primary"], row["rubric_names"], row["rubric_ids"]),
                         ("Кофейни", "Кондитерские;Кофейни", "164;162"))
        self.assertEqual((row["org_branch_count"], row["brand_branch_count"]), (7, 14))
        self.assertEqual((row["general_rating"], row["general_review_count"]), (4.7, 32))
        self.assertEqual((row["weekday_open_earliest"], row["opens_by_8"], row["open_weekend"], row["open_24x7"]),
                         ("07:30", True, True, False))
        self.assertTrue(row["temporary_closed"])
        self.assertEqual(row["building_name"], "Нурлы Тау, бизнес-центр")
        self.assertEqual((row["avg_check_kzt"], row["seats_max"]), (4500, 20))
        self.assertEqual((row["coffee_to_go"], row["laptop_friendly"], row["free_wifi"]), (True, True, False))
        self.assertIn("food_service_avg_price", row["attribute_tags"].split(";"))
        self.assertEqual(row["created_at"], "2012-04-18T12:08:03.273144+07:00")

    def test_parse_minimal_item_and_schedules(self):
        row = parse_item({"id": "5"})
        self.assertEqual((row["lon"], row["rubric_primary"], row["org_branch_count"]), (None, None, None))
        self.assertEqual((row["has_schedule"], row["opens_by_8"], row["temporary_closed"]), (False, False, False))
        self.assertEqual(parse_schedule({"is_24x7": True})["weekday_open_earliest"], "00:00")
        self.assertTrue(parse_schedule({"is_24x7": True})["opens_by_8"])
        late = parse_schedule({"Mon": {"working_hours": [{"from": "08:01", "to": "20:00"}]}})
        self.assertEqual((late["opens_by_8"], late["open_weekend"]), (False, False))
        self.assertFalse(parse_item({"id": "6", "flags": {"temporary_closed": "false"}})["temporary_closed"])
        self.assertIsNone(row["avg_check_kzt"])
        self.assertIsNone(parse_item({"id": "7", "reviews": {"general_rating": 0, "general_review_count": 3}})["general_rating"])
        check = lambda name: attribute_number([{"tag": "food_service_avg_price", "name": name}], "food_service_avg_price")
        self.assertEqual([check("Чек 8 000 ₸"), check("Чек 2000–4000\xa0₸"), check("Чек")], [8000, 3000, None])



suite = unittest.defaultTestLoader.loadTestsFromTestCase(EnrichTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("Проверки дозагрузки 2ГИС завершились ошибкой.")

test_bad_ids_are_rejected (__main__.EnrichTests.test_bad_ids_are_rejected) ... 

ok


test_batch_limit (__main__.EnrichTests.test_batch_limit) ... 

ok


test_batches_missing_ids_and_params (__main__.EnrichTests.test_batches_missing_ids_and_params) ... 

ok

... (11 outputs left out)

ok


test_server_error_message_hides_key (__main__.EnrichTests.test_server_error_message_hides_key) ... 

ok


----------------------------------------------------------------------
Ran 10 tests in 0.943s

OK


## Дозагрузка по ID

1. Ключ лежит в `.env` строкой `DGIS_API_KEY=...`. Ноутбук читает его только при `RUN_LIVE = True` и нигде не печатает.
2. **Пробный пакет.** Поставьте `RUN_LIVE = True` и выполните ячейку: уйдёт 1 запрос на 100 ID. Затем выполните «Разбор ответа» и проверьте, какие поля пришли и как называются рубрики. Если 2ГИС вернул ошибку с `meta.code` (например, поле недоступно для ключа), уберите это поле из `ENRICH_FIELDS` в `notebooks/dgis_api.py` и начните с новым файлом прогресса.
3. **Остальные.** Поставьте `BATCH_LIMIT = None` и `MAX_REQUESTS_TOTAL = 40`. Уже загруженные ID повторно не запрашиваются. Если позже появятся новые ID (например, после догрузки 18 точек), они допишутся новыми пакетами.

In [4]:
RUN_LIVE = False
BATCH_LIMIT = 1            # пробный пакет; затем None — все оставшиеся
MAX_REQUESTS_TOTAL = 5     # суммарный лимит HTTP-попыток с учётом уже потраченных
MIN_INTERVAL_SECONDS = 1.0

ids_to_enrich = [] if found is None else found["id"].tolist()
print(f"ID к дозагрузке: {len(ids_to_enrich)}; запросов без повторов: {-(-len(ids_to_enrich) // BYID_MAX_IDS)}")
print("Поля:", ", ".join(ENRICH_FIELDS))
if RUN_LIVE:
    if not ids_to_enrich:
        raise ValueError("Нет ID: положите файлы прогресса поиска в data/processed (см. раздел выше).")
    with requests.Session() as session:
        print(enrich(ids_to_enrich, ENRICH_CHECKPOINT, key=read_api_key(PROJECT_ROOT), get=session.get,
                     max_requests_total=MAX_REQUESTS_TOTAL, batch_limit=BATCH_LIMIT, interval=MIN_INTERVAL_SECONDS))
else:
    print("Запросы к 2ГИС выключены (RUN_LIVE = False).")

ID к дозагрузке: 7; запросов без повторов: 1
Поля: items.point, items.reviews, items.rubrics, items.org, items.brand, items.schedule, items.flags, items.address, items.attribute_groups, items.dates
Запросы к 2ГИС выключены (RUN_LIVE = False).


## Разбор ответа

Каждый сохранённый ответ раскладывается в строку таблицы. Координаты берутся из ответа byid, а если их там нет — из поиска. ID, которые 2ГИС не вернул, скорее всего закрыты или удалены: Places API ищет только по работающим организациям. В таблицу они не попадают, их число печатается отдельно.

После пробного пакета проверьте:
- какие поля реально пришли;
- как называются основные рубрики — по ним поправьте `COFFEE_RUBRICS`;
- значения `temporary_closed_raw` — тип этого флага в документации не описан;
- какие атрибуты встречаются (`attribute_tags`).

In [5]:
enriched = None
if ENRICH_CHECKPOINT.is_file():
    items, missing_ids, enrich_summary = load_enriched(ENRICH_CHECKPOINT)
    print(enrich_summary)
    print(f"Не вернулись: {len(missing_ids)} ID — вероятно, закрыты или удалены; в признаки не попадают")
    if items:
        print("Поле пришло у заведений (из", len(items), "):",
              {f: sum(f.split(".", 1)[1] in item for item in items) for f in ENRICH_FIELDS})
        enriched = pd.DataFrame([parse_item(item) for item in items])
        if found is not None:
            enriched = enriched.merge(found[["id", "lon", "lat", "source"]], on="id", how="left",
                                      suffixes=("", "_search"), validate="one_to_one")
            for col in ("lon", "lat"):
                enriched[col] = enriched[col].fillna(enriched[f"{col}_search"])
            enriched = enriched.drop(columns=["lon_search", "lat_search"])
        enriched = enriched.astype({"org_branch_count": "Int64", "brand_branch_count": "Int64",
                                    "general_review_count": "Int64", "avg_check_kzt": "Int64", "seats_max": "Int64"})
        enriched = classify(enriched, COFFEE_RUBRICS, CHAIN_MIN_BRANCHES).sort_values("id").reset_index(drop=True)
        assert enriched["id"].is_unique
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        enriched.to_csv(ENRICHED_CSV, index=False, lineterminator="\n")
        print(f"Сохранено: {ENRICHED_CSV.relative_to(PROJECT_ROOT)} ({len(enriched)} строк)")
        shares = enriched[["is_coffee", "is_chain", "opens_by_8", "open_24x7", "open_weekend", "temporary_closed",
                           "coffee_to_go", "free_wifi", "laptop_friendly"]].mean()
        print("Доли:", shares.round(3).to_dict())
        print("Средний чек, ₸:", enriched["avg_check_kzt"].describe().round(0).to_dict())
        display(enriched["rubric_primary"].value_counts(dropna=False).head(20).rename("заведений").to_frame())
        display(enriched["temporary_closed_raw"].value_counts(dropna=False).rename("заведений").to_frame())
        tags = enriched["attribute_tags"].str.split(";").explode()
        display(tags[tags.fillna("") != ""].value_counts().head(25).rename("заведений").to_frame())
else:
    print("Ответов byid ещё нет: запустите дозагрузку выше.")

{'attempts': 2, 'batches': {'pending': 0, 'done': 2, 'split': 0}, 'ids_requested': 7, 'items': 7, 'missing': 0, 'extra_items': 0}
Не вернулись: 0 ID — вероятно, закрыты или удалены; в признаки не попадают
Поле пришло у заведений (из 7 ): {'items.point': 7, 'items.reviews': 7, 'items.rubrics': 7, 'items.org': 7, 'items.brand': 1, 'items.schedule': 7, 'items.flags': 7, 'items.address': 7, 'items.attribute_groups': 7, 'items.dates': 7}
Сохранено: data\processed\2gis_enriched\organizations_enriched.csv (7 строк)
Доли: {'is_coffee': 1.0, 'is_chain': 0.429, 'opens_by_8': 0.571, 'open_24x7': 0.0, 'open_weekend': 1.0, 'temporary_closed': 0.0, 'coffee_to_go': 1.0, 'free_wifi': 0.286, 'laptop_friendly': 0.429}
Средний чек, ₸: {'count': 6.0, 'mean': 4500.0, 'std': 2049.0, 'min': 2000.0, '25%': 3375.0, '50%': 4500.0, '75%': 4875.0, 'max': 8000.0}


,заведений
rubric_primary,
Кофейни,7


,заведений
temporary_closed_raw,
None,7


,заведений
attribute_tags,
cocktail_bars_types_coffee_take_away,7
cocktail_bars_types_lemonade,7
cocktail_bars_types_tea_take_away,7
general_payment_type_card,7
general_payment_type_cash,7
general_payment_type_qrcode,7
cocktail_bars_types_cocoa,6
cocktail_bars_types_hot_chocolate,6
food_service_avg_price,6
